In [ ]:
print("\n"+"="*90); print("V6 vs COMPETITORS"); print("="*90)
print(f"\n{'Method':<24} {'Acc':>10} {'Params':>10} {'Uncertainty':>14} {'Ordinal':>12}")
print("-"*78)
print(f"{'MLTrMR (2025)':<24} {'80.19%':>10} {'556M':>10} {'None':>14} {'None':>12}")
print(f"{'LD2Net (2026)':<24} {'80.00%':>10} {'3.3M':>10} {'None':>14} {'None':>12}")

# Load CV results from saved file (safe to re-run independently)
cv_file = os.path.join(OUTPUT_DIR, "v6_phase5_cv.json")
if os.path.exists(cv_file):
    with open(cv_file) as f: cv_res = json.load(f)
    for mn,agg in cv_res["summary"].items():
        unc="Dirichlet" if mn in ("edl","edl_orcu") else "—"
        ord_t="Log-barrier" if mn=="edl_orcu" else ("Implicit" if mn=="edl" else "—")
        print(f"{'Ours '+mn.upper()+' (V6)':<24} "
              f"{agg['acc']['mean']:.2%}+/-{agg['acc']['std']:.2%}  "
              f"{'86M':>10} {unc:>14} {ord_t:>12}")
else:
    print("\n⚠ v6_phase5_cv.json not found — run Phase 5 first")

# Helper: convert numpy types for JSON serialization
def cvt(o):
    if isinstance(o,dict): return {k:cvt(v) for k,v in o.items()}
    elif isinstance(o,list): return [cvt(v) for v in o]
    elif isinstance(o,(np.floating,np.integer)): return float(o)
    elif isinstance(o,np.ndarray): return o.tolist()
    return o

# Master summary
all_s = {}
for fn,k in [("v6_phase2_baseline.json","p2"),("v6_phase3a_kl_sweep.json","p3a"),
             ("v6_phase3b_orcu_sweep.json","p3b"),("v6_phase4_multiseed.json","p4"),
             ("v6_phase5_cv.json","p5"),("v6_phase6_ablation.json","p6")]:
    fp=os.path.join(OUTPUT_DIR,fn)
    if os.path.exists(fp):
        with open(fp) as f: all_s[k]=json.load(f)
if all_s:
    with open(os.path.join(OUTPUT_DIR,"v6_master_summary.json"),"w") as f: json.dump(cvt(all_s),f,indent=2)
    print("\n=== V6 DONE ===")
    print("Master: v6_master_summary.json | CV: v6_phase5_cv.json | Ablation: v6_phase6_ablation.json")
else:
    print("\n⚠ No result files found — run experiment phases first")

# Export to Kaggle Dataset (if results exist)
import shutil
from pathlib import Path
import subprocess
DST="/kaggle/working/df_exp_result"
if os.path.exists(DST): shutil.rmtree(DST)
os.makedirs(DST,exist_ok=True)
exported = 0
for fn in ["v6_phase2_baseline.json","v6_phase3a_kl_sweep.json","v6_phase3b_orcu_sweep.json",
           "v6_phase4_multiseed.json","v6_phase5_cv.json","v6_phase6_ablation.json",
           "v6_master_summary.json"]:
    src=os.path.join(OUTPUT_DIR,fn)
    if os.path.exists(src):
        shutil.copy2(src,os.path.join(DST,fn))
        exported += 1
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".npz") and f.startswith("v6_"):
        shutil.copy2(os.path.join(OUTPUT_DIR,f),os.path.join(DST,f))
        exported += 1
if exported > 0:
    md={"title":"Fluorosis DF V6 Results","id":"hgxiao/fluorosis-df-v6-results","licenses":[{"name":"CC0-1.0"}]}
    with open(os.path.join(DST,"dataset-metadata.json"),"w") as f: json.dump(md,f)
    sz=sum(os.path.getsize(os.path.join(r,f)) for r,_,fs in os.walk(DST) for f in fs)
    print(f"\nExport: {sum(1 for _ in Path(DST).rglob('*') if _.is_file())} files, {sz/(1024*1024):.1f}MB")
    print("Upload: kaggle datasets create -p /kaggle/working/df_exp_result --dir-mode tar")
else:
    print("\n⚠ No files to export — run experiments first")

## Final Summary & Export

In [ ]:
# ===== Phase 2: Single-Seed Baseline =====
print("="*70); print("DF V6 Phase 2 — Single-Seed Baseline (s42)"); print("="*70)
phase2 = {}
for mode in MODES_V6:
    print(f"\n{'='*50}\n  {mode} | seed=42\n{'='*50}")
    out = os.path.join(OUTPUT_DIR, f"v6_p2_{mode}_s42")
    m, _ = train_model_v6("df", DATA_ROOT, out, mode=mode, epochs=EPOCHS, seed=42)
    phase2[mode] = {k:v for k,v in m.items() if isinstance(v,(float,int,bool,list))}
with open(os.path.join(OUTPUT_DIR,"v6_phase2_baseline.json"),"w") as f: json.dump(phase2,f,indent=2)
print("\n"+"="*70); print("PHASE 2 RESULTS"); print("="*70)
for mode in MODES_V6:
    m=phase2[mode]
    print(f"{mode:<14} Acc={m.get('acc',0):.4f} F1={m.get('macro_f1',0):.4f} QWK={m.get('qwk',0):.4f} ECE={m.get('ece',0):.4f}")
print("Phase 2 done.")

# ===== Phase 3A: EDL KL Sweep =====
print("\n"+"="*70); print("DF V6 Phase 3A — EDL KL Sweep"); print("="*70)
KL_VALS = [0.03, 0.05, 0.07, 0.10, 0.15]
phase3a = []
for kl in KL_VALS:
    for seed in SWEEP_SEEDS:
        print(f"\n  EDL kl={kl} s{seed}")
        out = os.path.join(OUTPUT_DIR, f"v6_p3a_edl_kl{kl}_s{seed}")
        m, _ = train_model_v6("df", DATA_ROOT, out, mode="edl", kl_lambda=kl, epochs=EPOCHS, seed=seed)
        r = {"kl_lambda":kl,"seed":seed,"acc":m["acc"],"macro_f1":m["macro_f1"],
             "qwk":m["qwk"],"ece":m["ece"],"pct_unimodal":m["pct_unimodal"]}
        if "per_class_f1" in m: r["per_class_f1"]=m["per_class_f1"]
        phase3a.append(r)
with open(os.path.join(OUTPUT_DIR,"v6_phase3a_kl_sweep.json"),"w") as f: json.dump(phase3a,f,indent=2)
for kl in KL_VALS:
    s=[r for r in phase3a if r["kl_lambda"]==kl]
    print(f"kl={kl:.2f}: Acc={np.mean([x['acc'] for x in s]):.4f}+/-{np.std([x['acc'] for x in s]):.4f}")
best=max(phase3a, key=lambda x:x["acc"])
print(f"Best: kl={best['kl_lambda']} s{best['seed']} Acc={best['acc']:.4f}")
print("Phase 3A done.")

# ===== Phase 3B: EDL+ORCU Lambda Sweep =====
print("\n"+"="*70); print("DF V6 Phase 3B — EDL+ORCU λ Sweep (log-barrier)"); print("="*70)
LAM_VALS = [0.05, 0.10, 0.20]
phase3b = []
for lam in LAM_VALS:
    for seed in SWEEP_SEEDS:
        print(f"\n  EDL+ORCU λ_reg={lam} (log-barrier) s{seed}")
        out = os.path.join(OUTPUT_DIR, f"v6_p3b_orcu_lam{lam}_s{seed}")
        m, _ = train_model_v6("df", DATA_ROOT, out, mode="edl_orcu",
                               lambda_reg=lam, use_log_barrier=True, epochs=EPOCHS, seed=seed)
        r = {"lambda_reg":lam,"seed":seed,"regularizer":"log_barrier",
             "acc":m["acc"],"macro_f1":m["macro_f1"],"qwk":m["qwk"],
             "ece":m["ece"],"pct_unimodal":m["pct_unimodal"]}
        if "per_class_f1" in m: r["per_class_f1"]=m["per_class_f1"]
        phase3b.append(r)
with open(os.path.join(OUTPUT_DIR,"v6_phase3b_orcu_sweep.json"),"w") as f: json.dump(phase3b,f,indent=2)
for lam in LAM_VALS:
    s=[r for r in phase3b if r["lambda_reg"]==lam]
    print(f"λ={lam:.2f}: Acc={np.mean([x['acc'] for x in s]):.4f}+/-{np.std([x['acc'] for x in s]):.4f}  "
          f"%Unim={np.mean([x['pct_unimodal'] for x in s]):.2%}")
print("Phase 3B done.")

# ===== Phase 4: 5-Seed Stability =====
print("\n"+"="*70); print("DF V6 Phase 4 — 5-Seed Stability"); print("="*70)
phase4 = {}
for mode in MODES_V6:
    phase4[mode] = {}
    for seed in SEEDS_V6:
        print(f"\n  {mode} | s{seed}")
        out = os.path.join(OUTPUT_DIR, f"v6_p4_{mode}_s{seed}")
        m, _ = train_model_v6("df", DATA_ROOT, out, mode=mode, epochs=EPOCHS, seed=seed)
        phase4[mode][str(seed)] = {k:v for k,v in m.items() if isinstance(v,(float,int,bool,list))}
with open(os.path.join(OUTPUT_DIR,"v6_phase4_multiseed.json"),"w") as f: json.dump(phase4,f,indent=2)
for mode in MODES_V6:
    accs=[phase4[mode][str(s)]['acc'] for s in SEEDS_V6]
    print(f"{mode:<14}: Acc={np.mean(accs):.4f}+/-{np.std(accs):.4f}")
print("Phase 4 done.")

# ===== Phase 5: 10-Fold CV =====
print("\n"+"#"*60); print("# DF V6 Phase 5: 10-Fold CV — PAPER RESULT"); print("#"*60)
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from scipy import stats

def run_cv_v6(data_root, methods_cfg, n_folds=10, epochs=100, seed=42):
    label_ds = DFDataset(data_root, split="train", split_seed=seed)
    labels = np.array([label_ds[i][1] for i in range(len(label_ds))])
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    mn_list = [m[0] for m in methods_cfg]
    fold_res = {mn: [] for mn in mn_list}

    for fi, (tr_idx, vl_idx) in enumerate(skf.split(np.zeros(len(label_ds)), labels)):
        print(f"\n{'='*50}\nFold {fi+1}/{n_folds}\n{'='*50}")
        tr_full = DFDataset(data_root, split="train", split_seed=seed)
        vl_full = DFDataset(data_root, split="train", split_seed=seed)
        tr_full.transform = get_transforms("df", is_train=True)
        vl_full.transform = get_transforms("df", is_train=False)
        tr_sub = Subset(tr_full, tr_idx); vl_sub = Subset(vl_full, vl_idx)
        test_ds = DFDataset(data_root, split="test")
        test_ds.transform = get_transforms("df", is_train=False)
        tr_ldr = DataLoader(tr_sub, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
        vl_ldr = DataLoader(vl_sub, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
        tst_ldr = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

        for mn, kw in methods_cfg:
            print(f"  [{mn}]")
            set_seed(seed + fi * 100)
            model = build_model(task="df", mode=mn); model.to(device)
            kl = kw.get("kl_lambda", V6_KL); kc = kw.get("kl_anneal_cap", V6_CAP)

            if mn == "ce":
                def cr(a,z,t,e): loss=F.nll_loss(F.log_softmax(z,dim=-1),t); return loss, {"stage":0,"L_ce":loss.item()}
            elif mn == "cumulative":
                from losses.cumulative_loss import CumulativeLoss
                cf=CumulativeLoss(num_classes=4)
                def cr(a,z,t,e):
                    ol=model.ordinal_logits if hasattr(model,'ordinal_logits') else z
                    loss=cf(ol,t); return loss, {"stage":0,"L_cum":loss.item()}
            elif mn == "sord":
                from losses.orcu_loss import ORCULoss
                sf=ORCULoss(num_classes=4,t=3.0,lambda_reg=0.0)
                def cr(a,z,t,e): loss=sf(z,t); return loss, {"stage":0,"L_sord":loss.item()}
            elif mn == "edl":
                from losses.edl_loss import EDLLoss
                ef=EDLLoss(num_classes=4,kl_lambda=kl,kl_anneal_cap=kc)
                def cr(a,z,t,e): loss=ef(a,t,e,epochs); return loss, {"stage":0,"L_edl":loss.item()}
            else:
                cr=CombinedLoss(num_classes=4,lambda_orcu=kw.get("lambda_orcu",0.10),
                                lambda_kl=kl,kl_anneal_cap=kc,orcu_t=3.0,
                                orcu_lambda_reg=kw.get("lambda_reg",0.10),
                                stage_1_epochs=3,stage_2_epochs=10,total_epochs=epochs)

            bb,hd=[],[]
            for n,p in model.named_parameters():
                if not p.requires_grad: continue
                (bb if n.startswith("backbone") else hd).append(p)
            opt=optim.AdamW([{"params":bb,"lr":1e-4},{"params":hd,"lr":1e-3}], weight_decay=0.05)
            ws=LinearLR(opt,start_factor=0.1,total_iters=5)
            cs=CosineAnnealingLR(opt,T_max=epochs-5)
            sch=SequentialLR(opt,schedulers=[ws,cs],milestones=[5])
            es=EarlyStopping(patience=kw.get("patience",15),mode="max")
            bva, bst = 0.0, None

            @torch.no_grad()
            def ev(m,ldr):
                m.eval(); aa,zz,tt=[],[],[]
                for im,tg in ldr:
                    im,tg=im.to(device),tg.to(device)
                    a,z_=m(im)
                    if a is not None: aa.append(a.cpu())
                    zz.append(z_.cpu()); tt.append(tg.cpu())
                a=torch.cat(aa) if aa else None
                return compute_metrics(a,torch.cat(zz),torch.cat(tt))

            for ep in range(epochs):
                model.train()
                for im,tg in tr_ldr:
                    im,tg=im.to(device),tg.to(device)
                    alpha,z=model(im); loss,_=cr(alpha,z,tg,ep)
                    opt.zero_grad(); loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
                    opt.step()
                sch.step()
                vm=ev(model,vl_ldr)
                if vm["acc"]>bva: bva=vm["acc"]; bst=copy.deepcopy(model.state_dict())
                if es(vm["acc"],ep,model): break

            es.restore(model); tm=ev(model,tst_ldr)
            fold_res[mn].append({"fold":fi,**{k:v for k,v in tm.items()
                if isinstance(v,(float,int,bool,list,np.ndarray))}})
            print(f"    Acc={tm['acc']:.4f} F1={tm['macro_f1']:.4f} QWK={tm['qwk']:.4f}")

    summary={}
    for mn in mn_list:
        agg={}
        for mk in ["acc","macro_f1","weighted_f1","qwk","ece","pct_unimodal"]:
            vals=[f[mk] for f in fold_res[mn] if mk in f]
            if vals: agg[mk]={"mean":float(np.mean(vals)),"std":float(np.std(vals))}
        if "per_class_f1" in fold_res[mn][0]:
            pf=np.array([f["per_class_f1"] for f in fold_res[mn]])
            agg["per_class_f1"]={"mean":pf.mean(0).tolist(),"std":pf.std(0).tolist()}
        summary[mn]=agg

    sig={}
    for i,m1 in enumerate(mn_list):
        for j,m2 in enumerate(mn_list):
            if i>=j: continue
            pk=f"{m1}_vs_{m2}"; sig[pk]={}
            for mk in ["acc","macro_f1","weighted_f1","qwk","ece"]:
                v1=[f[mk] for f in fold_res[m1]]; v2=[f[mk] for f in fold_res[m2]]
                t,p=stats.ttest_rel(v1,v2)
                sig[pk][mk]={"t_statistic":float(t),"p_value":float(p),"significant":bool(p<0.05)}
    return {"task":"df","k":n_folds,"methods":mn_list,"summary":summary,"significance":sig}

cv_cfg=[("ce",{"patience":15}), ("cumulative",{"patience":15}), ("sord",{"patience":15}),
        ("edl",{"kl_lambda":0.07,"kl_anneal_cap":0.5,"patience":25}),
        ("edl_orcu",{"kl_lambda":0.07,"lambda_orcu":0.10,"lambda_reg":0.10,"kl_anneal_cap":0.5,"patience":15})]
cv_res=run_cv_v6(DATA_ROOT, methods_cfg=cv_cfg, n_folds=10, epochs=EPOCHS)

def cvt(o):
    if isinstance(o,dict): return {k:cvt(v) for k,v in o.items()}
    elif isinstance(o,list): return [cvt(v) for v in o]
    elif isinstance(o,(np.floating,np.integer)): return float(o)
    elif isinstance(o,np.ndarray): return o.tolist()
    return o
with open(os.path.join(OUTPUT_DIR,"v6_phase5_cv.json"),"w") as f: json.dump(cvt(cv_res),f,indent=2)

print("\n"+"="*90); print("CV RESULTS — Paper Core"); print("="*90)
for mn,agg in cv_res["summary"].items():
    print(f"{mn:<14}: Acc={agg['acc']['mean']:.4f}+/-{agg['acc']['std']:.4f}  "
          f"QWK={agg['qwk']['mean']:.4f}+/-{agg['qwk']['std']:.4f}  "
          f"ECE={agg['ece']['mean']:.4f}")

edl_agg=cv_res["summary"].get("edl",{})
if edl_agg:
    a=edl_agg["acc"]["mean"]; q=edl_agg["qwk"]["mean"]
    tier = "S—Paper SOTA!" if a>=0.82 else ("A—Beats competitors!" if a>=0.80 else "B—Competitive")
    print(f"\nEDL CV: Acc={a:.4f} QWK={q:.4f} → Tier: {tier}")
print("Phase 5 done.")

# ===== Phase 6: A6 Ablation =====
print("\n"+"="*70); print("DF V6 Phase 6 — A6: Log-Barrier vs Hinge"); print("="*70)
phase6 = []
for reg_name, use_lb in [("log_barrier",True), ("hinge",False)]:
    for seed in SWEEP_SEEDS:
        print(f"\n  EDL+ORCU {reg_name} s{seed}")
        out = os.path.join(OUTPUT_DIR, f"v6_p6_{reg_name}_s{seed}")
        m, _ = train_model_v6("df", DATA_ROOT, out, mode="edl_orcu",
                               use_log_barrier=use_lb, epochs=EPOCHS, seed=seed)
        r = {"regularizer":reg_name,"seed":seed,
             "acc":m["acc"],"macro_f1":m["macro_f1"],"qwk":m["qwk"],
             "ece":m["ece"],"pct_unimodal":m["pct_unimodal"]}
        if "per_class_f1" in m: r["per_class_f1"]=m["per_class_f1"]
        phase6.append(r)
with open(os.path.join(OUTPUT_DIR,"v6_phase6_ablation.json"),"w") as f: json.dump(phase6,f,indent=2)

for reg in ["log_barrier","hinge"]:
    s=[r for r in phase6 if r["regularizer"]==reg]
    print(f"{reg:<16}: Acc={np.mean([x['acc'] for x in s]):.4f}  "
          f"%Unim={np.mean([x['pct_unimodal'] for x in s]):.2%}  "
          f"ECE={np.mean([x['ece'] for x in s]):.4f}")

lb_u=np.mean([r["pct_unimodal"] for r in phase6 if r["regularizer"]=="log_barrier"])
hg_u=np.mean([r["pct_unimodal"] for r in phase6 if r["regularizer"]=="hinge"])
print(f"\nΔ %Unimodal: {lb_u-hg_u:+.2%}  → {'✅ Log-barrier wins!' if lb_u>hg_u else '⚠ Check'}")
print("Phase 6 done.")

print("\n" + "="*90)
print("V6 EXPERIMENTS COMPLETE")
print("="*90)
print("\nKey files: v6_phase5_cv.json (paper) | v6_phase6_ablation.json (novelty)")
print("Download /kaggle/working/ for all outputs.")

## 4. Phase 2-6: All Experiments

### Phase 2: Single-Seed Baseline (5 runs, ~1.5h)
V6 defaults with seed=42 to verify no regressions.

In [ ]:
@torch.no_grad()
def evaluate_model(model, dataloader):
    model.eval()
    all_alpha, all_z, all_targets = [], [], []
    for images, targets in dataloader:
        images, targets = images.to(device), targets.to(device)
        alpha, z = model(images)
        if alpha is not None: all_alpha.append(alpha.cpu())
        all_z.append(z.cpu()); all_targets.append(targets.cpu())
    alpha = torch.cat(all_alpha) if all_alpha else None
    return compute_metrics(alpha, torch.cat(all_z), torch.cat(all_targets))


def train_model_v6(task, data_root, output_dir, mode="edl_orcu",
                   batch_size=32, epochs=100, lr_backbone=1e-4, lr_head=1e-3,
                   weight_decay=0.05, warmup_epochs=5,
                   kl_lambda=None, kl_anneal_cap=None,
                   lambda_orcu=0.10, lambda_reg=0.10, orcu_t=3.0,
                   stage_1_epochs=3, stage_2_epochs=10,
                   use_log_barrier=True, patience=None, seed=42):
    """V6 training: log-barrier ORCU, kl=0.07, 5-seed stability."""

    if patience is None: patience = 25 if mode == "edl" else 15
    if kl_lambda is None: kl_lambda = V6_KL
    if kl_anneal_cap is None: kl_anneal_cap = V6_CAP

    set_seed(seed)  # V6: CUDA determinism
    os.makedirs(output_dir, exist_ok=True)

    train_loader, val_loader, test_loader = create_dataloaders(
        data_root, task=task, batch_size=batch_size, num_workers=2)
    print(f"Data: train={len(train_loader.dataset)} val={len(val_loader.dataset)} "
          f"test={len(test_loader.dataset)}")

    model = build_model(task=task, mode=mode); model.to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {n_params:,} params | Mode: {mode} | Seed: {seed}")
    if mode in ("edl", "edl_orcu"):
        print(f"  V6: kl={kl_lambda} cap={kl_anneal_cap} | "
              f"log_barrier={use_log_barrier} lambda_reg={lambda_reg}")

    # ---- Criterion (mode-dependent) ----
    if mode == "ce":
        def crit(a, z, t, e):
            loss = F.nll_loss(F.log_softmax(z, dim=-1), t); return loss, {"stage":0,"L_ce":loss.item()}
    elif mode == "cumulative":
        from losses.cumulative_loss import CumulativeLoss
        cf = CumulativeLoss(num_classes=4)
        def crit(a, z, t, e):
            ol = model.ordinal_logits if hasattr(model,'ordinal_logits') else z
            loss = cf(ol, t); return loss, {"stage":0,"L_cum":loss.item()}
    elif mode == "sord":
        from losses.orcu_loss import ORCULoss
        sf = ORCULoss(num_classes=4, t=orcu_t, lambda_reg=0.0)
        def crit(a, z, t, e):
            loss = sf(z, t); return loss, {"stage":0,"L_sord":loss.item()}
    elif mode == "edl":
        from losses.edl_loss import EDLLoss
        ef = EDLLoss(num_classes=4, kl_lambda=kl_lambda, kl_anneal_cap=kl_anneal_cap)
        def crit(a, z, t, e):
            loss = ef(a, t, e, epochs); return loss, {"stage":0,"L_edl":loss.item()}
    else:  # edl_orcu
        crit = CombinedLoss(
            num_classes=4, lambda_orcu=lambda_orcu, lambda_kl=kl_lambda,
            kl_anneal_cap=kl_anneal_cap, orcu_t=orcu_t,
            orcu_lambda_reg=lambda_reg,
            stage_1_epochs=stage_1_epochs, stage_2_epochs=stage_2_epochs,
            total_epochs=epochs)

    # ---- Optimizer & Scheduler ----
    bb, hd = [], []
    for n, p in model.named_parameters():
        if not p.requires_grad: continue
        (bb if n.startswith("backbone") else hd).append(p)
    optimizer = optim.AdamW([{"params":bb,"lr":lr_backbone},
                              {"params":hd,"lr":lr_head}], weight_decay=weight_decay)
    ws = LinearLR(optimizer, start_factor=0.1, total_iters=warmup_epochs)
    cs = CosineAnnealingLR(optimizer, T_max=epochs - warmup_epochs)
    scheduler = SequentialLR(optimizer, schedulers=[ws,cs], milestones=[warmup_epochs])

    # ---- Training Loop ----
    es = EarlyStopping(patience=patience, mode="max")
    best_val_acc, best_epoch = 0.0, 0; stopped_early = False
    history = []

    for epoch in range(epochs):
        model.train()
        el = {"L_ce":0,"L_cum":0,"L_sord":0,"L_edl":0,"L_orcu":0,"L_total":0}
        epoch_stage, nb = 0, 0

        for images, targets in train_loader:
            images, targets = images.to(device), targets.to(device)
            alpha_out, z = model(images)
            loss, loss_info = crit(alpha_out, z, targets, epoch)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # V6
            optimizer.step()

            epoch_stage = loss_info.get("stage",0)
            for k in el:
                if k in loss_info: el[k] += loss_info[k]
            nb += 1

        scheduler.step()
        for k in el: el[k] /= max(nb, 1)

        vm = evaluate_model(model, val_loader)
        if vm["acc"] > best_val_acc:
            best_val_acc = vm["acc"]; best_epoch = epoch

        history.append({"epoch":epoch,"stage":epoch_stage,"train_loss":el,
                        "val_metrics":{k:v for k,v in vm.items()
                         if isinstance(v,(float,int)) and not k.startswith("evidence_class_")}})

        lk = next((k for k in ["L_total","L_ce","L_cum","L_sord","L_edl","L_orcu"]
                   if el.get(k,0)!=0), "L_total")
        if (epoch+1)%10==0 or epoch==0 or epoch==epochs-1:
            print(f"[E{epoch+1:3d}/{epochs}] S={epoch_stage} | {lk}={el.get(lk,0):.4f} | "
                  f"Acc={vm['acc']:.4f} F1={vm['macro_f1']:.4f} "
                  f"QWK={vm['qwk']:.4f} ECE={vm['ece']:.4f} ES={es.counter}/{patience}")

        if es(vm["acc"], epoch, model):
            print(f"  EarlyStop@{epoch+1} (best: ep{best_epoch+1} acc={best_val_acc:.4f})")
            stopped_early = True; break

    if not stopped_early:
        print(f"  Done {epochs} ep (best: ep{best_epoch+1} acc={best_val_acc:.4f})")

    es.restore(model)
    tm = evaluate_model(model, test_loader)

    # Save
    torch.save({"model_state_dict":model.state_dict(),"test_metrics":tm,
                "best_epoch":best_epoch,"stopped_early":stopped_early},
               os.path.join(output_dir,"best.pt"))

    ser = {}
    for k,v in tm.items():
        if isinstance(v,(float,int,bool)): ser[k]=v
        elif isinstance(v,(np.floating,np.integer)): ser[k]=float(v)
        elif isinstance(v,np.ndarray): ser[k]=v.tolist()
        elif isinstance(v,list): ser[k]=v
    with open(os.path.join(output_dir,"test_metrics.json"),"w") as f:
        json.dump(ser,f,indent=2)
    with open(os.path.join(output_dir,"history.json"),"w") as f:
        json.dump(history,f,indent=2)

    # Per-sample export
    from eval import export_predictions, save_predictions
    pred = export_predictions(model, test_loader, device)
    save_predictions(pred, os.path.join(output_dir,f"v6_{mode}_s{seed}_preds.npz"),
                     mode=mode, task=task, seed=seed)

    print(f"\n[Test] Acc={tm['acc']:.4f} F1={tm['macro_f1']:.4f} "
          f"WgtF1={tm['weighted_f1']:.4f} QWK={tm['qwk']:.4f} "
          f"ECE={tm['ece']:.4f} BestEp={best_epoch+1} ES={stopped_early}")
    if "per_class_f1" in tm:
        p=tm["per_class_f1"]
        print(f"  Per-Class F1: N={p[0]:.4f} Mi={p[1]:.4f} Mo={p[2]:.4f} Se={p[3]:.4f}")
    return tm, history

print("train_model_v6() ready.")
print(f"V6 defaults: epochs={EPOCHS} kl={V6_KL} cap={V6_CAP} lambda_reg={V6_LAMBDA_REG} log_barrier=True")

## 3. V6 Training Function (log-barrier ORCU + kl=0.07)

Key changes from V5:
- `set_seed()` instead of `torch.manual_seed()` (CUDA determinism)
- `EarlyStopping` from utils.py (in-memory best model)
- `clip_grad_norm_(max_norm=1.0)` for gradient stability
- `use_log_barrier=True` (False for A6 hinge ablation)

In [ ]:
# Master Setup — cache ViT weights while internet is available
import os, sys, json, copy, itertools
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F, torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

print("Caching ViT-Base ImageNet-21k weights...")
from transformers import ViTModel
_ = ViTModel.from_pretrained("google/vit-base-patch16-224-in21k")
print("Weights cached.")

CODE_ROOT = "/kaggle/working/fluorosis_project/06_Implementation"
DATA_ROOT = "/kaggle/input/datasets/hgxiao/fluorosis_df_pro"
OUTPUT_DIR = "/kaggle/working"
sys.path.insert(0, CODE_ROOT)

from data import create_dataloaders, DFDataset, get_transforms
from models import build_model
from losses import CombinedLoss
from eval import compute_metrics
from utils import set_seed, EarlyStopping  # V6: new utility module

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
print("V6 setup complete.")

# ===== V6 Constants =====
EPOCHS = 100
SEEDS_V6 = [42, 123, 456, 789, 1024]       # V6: 5 seeds
SWEEP_SEEDS = [42, 123, 456]                # 3 seeds for sweeps
MODES_V6 = ["ce", "cumulative", "sord", "edl", "edl_orcu"]
V6_KL = 0.07        # V6 unified kl_lambda
V6_CAP = 0.5         # KL anneal cap
V6_LAMBDA_REG = 0.10 # EDL+ORCU regularization (log-barrier)
print(f"V6: kl={V6_KL} cap={V6_CAP} lambda_reg={V6_LAMBDA_REG} log_barrier=True")

In [ ]:
!pip install transformers scikit-learn openpyxl scipy -q

# Clone updated V6 code from GitHub
!rm -rf /kaggle/working/fluorosis_project
!git clone https://github.com/XiaoHG/fluorosis_project.git /kaggle/working/fluorosis_project

import os
assert os.path.isdir("/kaggle/working/fluorosis_project/06_Implementation"), "Git clone failed!"
print("V6 code cloned successfully.")

## 1. Environment Setup

# V6.0 DF Experiment — Log-Barrier ORCU + KL Fix + 10-Fold CV + 5-Seed

**V6 fixes from audit (2026-06-01):**
- **ORCU**: hinge → **log-barrier** (matches Kim et al. design doc, `t` param now active)
- **EDL kl_lambda**: 0.05 → **0.07** (fix V5 KL annealing regression)
- **lambda_kl unified**: all sources now 0.07 (was inconsistent: 0.02/0.1/0.1/0.1)
- **EDL+ORCU lambda_reg**: 0.005 → **0.10** (stronger with smoother log-barrier)
- **Seeds**: 3 → **5** (42, 123, 456, 789, 1024) for robust stability
- **Infrastructure**: EarlyStopping (utils.py), CUDA determinism (set_seed), gradient clipping, AMP — all done

**Experiment (104 runs, ~25h on T4 x2):**
| Phase | Content | Runs | Time | Priority |
|-------|---------|------|------|----------|
| 2 | Single-seed baseline (5 methods × s42) | 5 | ~1.5h | HIGH |
| 3A | EDL kl sweep [.03,.05,.07,.10,.15] × 3 seeds | 9 | ~2.5h | HIGH |
| 3B | EDL+ORCU λ sweep [.05,.10,.20] × 3 seeds | 9 | ~2.5h | HIGH |
| 4 | Multi-seed (5 methods × 5 seeds) | 25 | ~5h | HIGH |
| 5 | **10-fold CV (5 methods)** | **50** | **~15h** | CRITICAL |
| 6 | A6 ablation (log-barrier vs hinge) × 3 seeds | 6 | ~1h | MEDIUM |

**Success: Acc > 80% → beats MLTrMR (80.19%) and LD2Net (80.00%)**